# Inspect Aggregated Recharge Datasets

HRU-level inspection of the three recharge sources. Mirrors `inspect_consolidated_recharge.ipynb`.

Sources (see `catalog/variables.yml` → `recharge`):

- Reitz 2017 (`total_recharge`, m/year, annual).
- WaterGAP 2.2d (`qrdif`, kg m⁻² s⁻¹ rate, monthly mean).
- ERA5-Land (`ssro`, m/month, monthly accumulation) — sub-surface runoff used as a recharge proxy.

See `docs/references/calibration-target-recipes.md` §3 for canonical-unit conversions and the per-source 0–1 normalisation that makes them combinable.

## Per-target conventions in this notebook

- Reitz `total_recharge` is native m/year (NOT inches/year — earlier catalog versions had the wrong units; recipes §3 / cross-cutting lesson 1). × 1000 → mm/year.
- WaterGAP `qrdif` is kg m⁻² s⁻¹ rate. For each month, × `(days_in_month × 86400)` → mm/month. Sum 12 months → mm/year.
- ERA5-Land `ssro` is m/month accumulated. × 1000 → mm/month. Sum 12 months → mm/year.
- **Cell 7 shows native-unit maps** at native cadence (Reitz annual mid-year, WaterGAP/ERA5-Land monthly Jan).
- **Cells 9–13 operate at annual mm/year** — the cadence the target builder will combine on.
- Reference source for the colour scale: ERA5-Land.

In [ ]:
import calendar
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    discover_aggregated,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    nan_hru_count,
    open_year,
    open_year_range,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    select_month,
    unit_from_catalog,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/inspect_aggregated/<run>_*.png
# import _helpers
# _helpers.SAVE_FIGURES = True

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)

TARGET = "recharge"
TARGET_YEAR = 2000
TARGET_TIME = f"{TARGET_YEAR} (annual)"
TARGET_MONTH = 1
TIME_SERIES_YEARS = range(2000, 2010)
REPRESENTATIVE_POINTS = {
    "Olympic Peninsula (PNW mountains)": (-123.5, 47.8),
    "Iowa cropland (Midwest)": (-93.6, 42.0),
    "Phoenix metro (arid SW)": (-112.1, 33.4),
    "Southern Appalachians (Eastern forest)": (-83.5, 35.5),
}

SOURCES = {
    "reitz2017": {"label": "Reitz 2017 (total_recharge)", "var": "total_recharge"},
    "watergap22d": {"label": "WaterGAP 2.2d (qrdif)", "var": "qrdif"},
    "era5_land": {"label": "ERA5-Land (ssro)", "var": "ssro"},
}

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
print(f"Fabric: {fabric_cfg['path']} ({len(fabric)} HRUs)")

## Load aggregated datasets

Each source is opened from `<project>/data/aggregated/<source_key>/<source_key>_<TARGET_YEAR>_agg.nc`. Sources whose aggregation has not yet been produced are skipped with a clear message; downstream cells iterate over the loaded set so missing sources drop out naturally.

In [ ]:
opened = {}
for source_key, info in SOURCES.items():
    paths = discover_aggregated(project_dir, source_key)
    if paths is None:
        print(f"SKIP {info['label']}: no aggregated NCs at "
              f"{project_dir}/data/aggregated/{source_key}/")
        continue
    ds = open_year(project_dir, source_key, TARGET_YEAR)
    info["units"] = unit_from_catalog(source_key, info["var"])
    opened[source_key] = (ds, info)
    values = ds[info["var"]].isel(time=0).to_pandas()
    print(
        f"{info['label']}: vars={list(ds.data_vars)} | "
        f"time={ds.time.values[0]}..{ds.time.values[-1]} | "
        f"HRUs={ds.sizes.get('nhm_id', ds.sizes.get('hru_id', 'unknown'))} | "
        f"NaN HRUs (t=0): {nan_hru_count(values)} | "
        f"catalog units: {info['units']}"
    )

In [ ]:
for source_key, (ds, info) in opened.items():
    print(f"{'=' * 60}\n{info['label']}\n{'=' * 60}")
    display(ds)

In [ ]:
def _to_mm_per_year(ds: xr.Dataset, source_key: str, var: str, year: int) -> "pd.Series":
    """Return per-HRU annual recharge in mm/year for the given calendar year.

    Reitz: annual native, × 1000.
    WaterGAP: monthly rate × (days_in_month × 86400), summed over the year.
    ERA5-Land: monthly accumulated × 1000, summed over the year.
    """
    if source_key == "reitz2017":
        da_annual = ds[var].sel(time=str(year), method="nearest")
        return (da_annual * 1000.0).to_pandas()
    if source_key == "watergap22d":
        da = ds[var].sel(time=str(year))
        days = pd.DatetimeIndex(da.time.values).days_in_month
        seconds = xr.DataArray(days.values * 86400.0, coords={"time": da.time}, dims=["time"])
        mm_per_month = da * seconds
        return mm_per_month.sum("time", skipna=False).to_pandas()
    if source_key == "era5_land":
        da = ds[var].sel(time=str(year))
        mm_per_month = da * 1000.0
        return mm_per_month.sum("time", skipna=False).to_pandas()
    raise ValueError(f"No annual conversion for {source_key}")


if not opened:
    print("No sources available; skipping native-unit map.")
else:
    fig, axes = plt.subplots(1, len(opened), figsize=(8 * len(opened), 6), squeeze=False)
    for idx, (source_key, (ds, info)) in enumerate(opened.items()):
        if source_key == "reitz2017":
            da = ds[info["var"]].sel(time=str(TARGET_YEAR), method="nearest")
            label_time = f"{TARGET_YEAR} (annual)"
        else:
            da = select_month(ds[info["var"]], TARGET_YEAR, TARGET_MONTH)
            label_time = f"{TARGET_YEAR}-01"
        plot_hru_choropleth(
            axes[0, idx],
            fabric,
            da.to_pandas(),
            cmap="YlGnBu",
            title=f"{info['label']}\n{label_time} | {info['units']}",
            units=info["units"],
        )
    fig.suptitle("Recharge \u2014 native units (Reitz annual, others monthly)", fontsize=13, y=1.02)
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_native_units_map")
    plt.show()

## Normalized comparison map (mm/year)

Sum each source’s monthly contributions (or pass through, for Reitz’s already-annual values) to **mm/year** and plot side by side on a shared colour scale derived from ERA5-Land (the reference source).

- Reitz `total_recharge` (m/year) × 1000 → mm/year.
- WaterGAP `qrdif` (kg m⁻² s⁻¹ rate) × `days_in_month × 86400` per month, summed across 12 months → mm/year.
- ERA5-Land `ssro` (m/month accumulated) × 1000 per month, summed across 12 months → mm/year.

This is the cadence the target builder will combine on. The three sources measure different fluxes (Reitz: empirical total recharge; WaterGAP: process-modelled diffuse recharge; ERA5-Land ssro: sub-surface runoff), so absolute magnitudes diverge — the target’s per-source 0–1 normalisation handles that.

In [ ]:
if opened:
    annual = {
        source_key: _to_mm_per_year(ds, source_key, info["var"], TARGET_YEAR)
        for source_key, (ds, info) in opened.items()
    }

    ref_key = "era5_land"
    if ref_key in annual:
        ref_vals = annual[ref_key].dropna().values
        vmin, vmax = float(np.percentile(ref_vals, 2)), float(np.percentile(ref_vals, 98))
    else:
        all_vals = np.concatenate([s.dropna().values for s in annual.values()])
        vmin, vmax = float(np.percentile(all_vals, 2)), float(np.percentile(all_vals, 98))

    fig, axes = plt.subplots(1, len(annual), figsize=(8 * len(annual), 6), squeeze=False)
    for idx, (source_key, values) in enumerate(annual.items()):
        plot_hru_choropleth(
            axes[0, idx],
            fabric,
            values,
            vmin=vmin,
            vmax=vmax,
            cmap="YlGnBu",
            title=f"{SOURCES[source_key]['label']}\n{TARGET_YEAR} | mm/year",
            units="mm/year",
        )
    fig.suptitle(
        f"Recharge \u2014 mm/year, colour scale from ERA5-Land \u2014 {TARGET_YEAR}",
        fontsize=13, y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_normalized_comparison")
    plt.show()

In [ ]:
if opened:
    fig, ax = plt.subplots(figsize=(10, 5))
    for source_key, values in annual.items():
        ax.hist(
            values.dropna(),
            bins=60,
            histtype="step",
            label=SOURCES[source_key]["label"],
            density=True,
            linewidth=2,
        )
    ax.set_xlabel("Recharge (mm/year)")
    ax.set_ylabel("Density")
    ax.set_title(f"Cross-source HRU distribution \u2014 {TARGET_YEAR}")
    ax.legend()
    save_figure(fig, f"{TARGET}_histogram")
    plt.show()

In [ ]:
if opened:
    rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
    print("Representative HRUs:", rep_hrus)

    series_data = {}
    id_dim = None
    for source_key, info in SOURCES.items():
        if source_key not in opened:
            continue
        ds_range = open_year_range(project_dir, source_key, TIME_SERIES_YEARS)
        try:
            id_dim = "nhm_id" if "nhm_id" in ds_range.dims else "hru_id"
            sel = ds_range[info["var"]].sel({id_dim: list(rep_hrus.values())}).load()
        finally:
            ds_range.close()
        series_data[source_key] = sel

    def _convert_series_annual(da, source_key):
        """Aggregate a 1-D (time,) per-HRU series to mm/year per calendar year."""
        years = pd.DatetimeIndex(da.time.values).year
        out_years = sorted(set(years))
        rows = []
        for y in out_years:
            mask = years == y
            year_da = da.isel(time=np.where(mask)[0])
            if source_key == "reitz2017":
                rows.append(year_da.mean("time").values * 1000.0)
            elif source_key == "watergap22d":
                days = pd.DatetimeIndex(year_da.time.values).days_in_month
                seconds = xr.DataArray(
                    days.values * 86400.0, coords={"time": year_da.time}, dims=["time"]
                )
                rows.append((year_da * seconds).sum("time").values)
            elif source_key == "era5_land":
                rows.append((year_da * 1000.0).sum("time").values)
        return pd.DataFrame(rows, index=out_years)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
        for source_key, da in series_data.items():
            df = _convert_series_annual(da.sel({id_dim: hru_id}), source_key)
            ax.plot(df.index, df.iloc[:, 0], marker="o", label=SOURCES[source_key]["label"])
        ax.set_title(f"{label} (HRU {hru_id})")
        ax.set_ylabel("mm/year")
        ax.legend(fontsize=8)
    fig.suptitle(
        f"Recharge at representative HRUs \u2014 {min(TIME_SERIES_YEARS)}\u2013{max(TIME_SERIES_YEARS)} (annual)"
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_time_series")
    plt.show()

In [ ]:
if opened:
    fig, axes = plt.subplots(1, len(opened), figsize=(8 * len(opened), 6), squeeze=False)
    for idx, (source_key, (ds, info)) in enumerate(opened.items()):
        values = annual[source_key]
        n_nan = nan_hru_count(values)
        print(f"{info['label']}: {n_nan} NaN HRUs ({100 * n_nan / len(fabric):.2f}%)")
        plot_nan_hrus(
            axes[0, idx],
            fabric,
            values,
            title=f"{info['label']}\nNaN HRUs (red) \u2014 {n_nan} of {len(fabric)}",
        )
    fig.suptitle(
        "Coverage gaps (annual mm/year) \u2014 these will be NN-filled in normalize/",
        fontsize=12,
        y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_coverage")
    plt.show()

In [ ]:
def _gridded_mean_recharge(source_key, info):
    """Compute the gridded CONUS-footprint mean for the validation cell.

    Approximate cross-check, not a like-for-like comparison: gridded mean
    is unweighted over the consolidated-NC bbox (which extends past the
    fabric into ocean/Canada/Mexico edges); HRU mean is Albers-area-
    weighted over fabric HRUs only. Differences of 5-30% are normal for
    sources with significant out-of-fabric coverage.
    """
    if source_key == "reitz2017":
        path = datastore_dir / "reitz2017" / "reitz2017.nc"
    elif source_key == "watergap22d":
        path = datastore_dir / "watergap22d" / "watergap22d.nc"
    elif source_key == "era5_land":
        path = datastore_dir / "era5_land" / "monthly" / f"era5_land_monthly_{TARGET_YEAR}.nc"
    else:
        return None, f"unknown gridded path for {source_key}"
    if not path.exists():
        return None, f"missing consolidated NC at {path}"
    with xr.open_dataset(path) as ds:
        if source_key == "reitz2017":
            da = ds[info["var"]].sel(time=str(TARGET_YEAR), method="nearest").load()
            converted = da * 1000.0
        elif source_key == "watergap22d":
            da = ds[info["var"]].sel(time=str(TARGET_YEAR)).load()
            days = pd.DatetimeIndex(da.time.values).days_in_month
            seconds = xr.DataArray(days.values * 86400.0, coords={"time": da.time}, dims=["time"])
            converted = (da * seconds).sum("time", skipna=False)
        else:  # era5_land
            da = ds[info["var"]].sel(time=str(TARGET_YEAR)).load()
            converted = (da * 1000.0).sum("time", skipna=False)
    return float(converted.mean(skipna=True).item()), None


print(f"{'Source':<35} {'Aggregated':>12} {'Gridded':>12} {'\u0394':>12} {'% diff':>8}")
print("-" * 85)
for source_key, (ds, info) in opened.items():
    agg_mean = area_weighted_mean(annual[source_key], fabric)
    gridded_mean, reason = _gridded_mean_recharge(source_key, info)
    if gridded_mean is None:
        print(f"{info['label']:<35} {agg_mean:>12.3f}  {'SKIP':>12} ({reason})")
        continue
    diff = agg_mean - gridded_mean
    pct = 100 * diff / gridded_mean if gridded_mean else float("nan")
    print(f"{info['label']:<35} {agg_mean:>12.3f} {gridded_mean:>12.3f} {diff:>12.3f} {pct:>7.2f}%")

## Why HRU-level patterns differ across sources

After area-weighted aggregation to HRUs, the HRU-level magnitudes are within the same order of magnitude as the gridded means (validation cell above). The validation cell’s “gridded” column is an unweighted mean over the consolidated-NC bbox; the “aggregated” column is an Albers-area-weighted mean over fabric HRUs only. A 5–30% difference between the two columns is normal.

Beyond that geographic mismatch, the three recharge sources measure conceptually different fluxes:

- **Reitz 2017** — empirical regression estimate of total groundwater recharge over CONUS at ~1 km. Annual cadence. Earlier catalog versions said `inches/year`; correct units are `m/year`.
- **WaterGAP 2.2d** — process-modelled diffuse groundwater recharge (`qrdif`). 0.5° global grid, monthly mean rate.
- **ERA5-Land `ssro`** — sub-surface runoff (drainage out the bottom of the soil column). Used as a proxy, not formally equivalent to recharge.

Absolute magnitudes diverge — particularly in arid regions, where Reitz can be ~0.1 in/yr and WaterGAP / ERA5-Land may differ by an order of magnitude. The target builder normalises each source 0–1 over the calibration window (catalog default 2000–2009 per TM 6-B10 body text) before computing per-HRU `min/max`. The optimiser then targets relative year-to-year change, not absolute magnitude — which is what makes these three combinable.

**Calibration target implication.** The 0–1 normalisation is doing the heavy lifting; absolute-magnitude validation in this notebook (cell 13) is a unit-conversion sanity check, not a comparability claim.

In [ ]:
for source_key, (ds, _) in opened.items():
    ds.close()
opened.clear()